# Entrenamiento del modelo final

Reproduce el modelo final del TP3 (`19_atributos_categoria_titulo`, accuracy anidada 0,9473, Kaggle 0,94690): un promedio ponderado de tres modelos.

| Modelo | Variables | Origen |
|---|---|---|
| LightGBM | 164 | se entrena acá |
| CatBoost con ids crudos (`seller_id`, `category_id`, ciudad) | 123 | se entrena acá |
| TabICLv2 (top 100 variables) | 100 | predicciones guardadas (Kaggle, GPU) |

Todo sale de los jsonlines de `datos/`, salvo las predicciones de los modelos entrenados en GPU (`predicciones_gpu/`): los tres fine-tunings de BETO, que entran como variable del LightGBM, y TabICLv2.

**Validación.** Siempre los mismos 5 folds estratificados (`random_state=42`). Cada modelo se corre dos veces:
- **CV anidada:** las variables que usan el target se calculan, para cada fold, solo con su entrenamiento (con folds internos). Sus predicciones out-of-fold sirven para elegir los pesos del promedio.
- **CV estándar:** las variables con target de train son out-of-fold y las de test se calculan con todo train. La predicción de test es el promedio de los 5 modelos de los folds.

Tarda alrededor de 85 minutos en CPU (casi todo es el CatBoost y los submodelos de texto anidados). Después, `2_prediccion.ipynb` aplica los modelos guardados a test.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display

from features import (COLUMNAS_CATEGORIA_SVD, agregar_features_duracion, agregar_features_fechas, agregar_features_texto,
                      categorias_por_titulos, leer_fotos_atributos, leer_json_crudo, normalizar_texto, valores_campos_json,
                      variables_atributos, variables_fotos, variables_patrones, variables_vendedor_categoria)
from modelado import *

import modelado
assert Path(modelado.__file__).resolve().parent == Path("src").resolve(), "Ejecutar el notebook desde la carpeta entrega/"
CONF = cargar_configuracion()
COLUMNAS_LGBM = CONF["lightgbm"]["columnas"]
COLUMNAS_CB = CONF["catboost"]["columnas"]
IDS_CRUDOS = CONF["catboost"]["ids_crudos"]
M = CONF["m_target_encoding"]
tiempos = {}
inicio_total = time.time()
print(f"LightGBM: {len(COLUMNAS_LGBM)} variables | CatBoost: {len(COLUMNAS_CB)} variables")

LightGBM: 164 variables | CatBoost: 123 variables


## a) Carga y procesamiento

`procesar()` (en `src/procesamiento.py`) aplica a train y a test las mismas transformaciones de limpieza del análisis exploratorio. `preparar_datos()` arma las matrices: `y` = 1 si la publicación es usada, las booleanas pasan a 0/1 y los textos a categorías con las categorías de train.

In [2]:
inicio = time.time()
X, y, X_test, ids_test, df_train, df_test = preparar_datos(
    agregar_features=lambda df: agregar_features_duracion(agregar_features_fechas(agregar_features_texto(df))),
    devolver_procesados=True)
Y = y.to_numpy()
folds = obtener_folds(y)
FOLD = np.empty(len(Y), dtype=int)
for k, (_, idx_val) in enumerate(folds, start=1):
    FOLD[idx_val] = k
tiempos["a) carga y procesamiento"] = time.time() - inicio
print(f"Train: {X.shape} | test: {X_test.shape} | % used en train: {Y.mean():.3f}")

Train: (70000, 93) | test: (30000, 93) | % used en train: 0.463


## b) Feature engineering

Variables sin target. Las que necesitan otras filas (frecuencias, perfil del vendedor, categoría descrita por sus títulos) se calculan sobre train + test, porque no usan `condition`.

- **Texto, fechas y duración** (ya agregadas en a): booleanas de palabras del título y de la garantía y métricas del título; días hasta la última actualización, día y hora de creación; duración de la publicación (las que duran 60 días y unos segundos son casi todas usadas).
- **Vendedor y categoría:** frecuencias de vendedor, categoría y ciudad, perfil del vendedor (stock, tipo de publicación, precio, fotos...) y precio relativo a la mediana de la categoría.
- **Fotos y atributos** (del JSON original): antigüedad de las fotos según su id y valores de atributos (km y año de vehículos, antigüedad de inmuebles, etc.).
- **Patrones:** 22 valores del JSON con mucha señal encontrados en el análisis de errores (modelo 12).
- **Categoría descrita por sus títulos:** un documento por categoría con todos sus títulos, TF-IDF y SVD de 16 componentes (`cat_svd_1..16`); también da las 10 categorías más parecidas de cada una, que se usan en c).
- **Ids crudos para el CatBoost:** `seller_id`, `category_id` y ciudad como texto. El CatBoost los codifica internamente, así que no recibe los target encodings del vendedor y la categoría.

In [3]:
inicio = time.time()
variables_train, variables_test = variables_vendedor_categoria(df_train, df_test)
X, X_test = X.join(variables_train), X_test.join(variables_test)

datos_train, datos_test = leer_fotos_atributos(ARCHIVO_TRAIN), leer_fotos_atributos(ARCHIVO_TEST)
datos_train.index, datos_test.index = X.index, X_test.index
atributos_train, atributos_test = variables_atributos(datos_train, datos_test)
X = X.join(variables_fotos(datos_train)).join(atributos_train)
X_test = X_test.join(variables_fotos(datos_test)).join(atributos_test)

PATRONES = [tuple(p.split(": ", 1)) for p in CONF["patrones"]]
crudo_train, crudo_test = leer_json_crudo(ARCHIVO_TRAIN), leer_json_crudo(ARCHIVO_TEST)
X = X.join(variables_patrones(valores_campos_json(crudo_train), PATRONES).set_index(X.index))
X_test = X_test.join(variables_patrones(valores_campos_json(crudo_test), PATRONES).set_index(X_test.index))

CAT_SVD, CAT_VECINAS, _ = categorias_por_titulos(df_train, df_test)
X = X.join(CAT_SVD.reindex(df_train["category_id"].astype(str)).set_index(X.index))
X_test = X_test.join(CAT_SVD.reindex(df_test["category_id"].astype(str)).set_index(X_test.index))

# Matrices del CatBoost: con los ids crudos y las categóricas como texto
X_cb, X_test_cb = X.copy(), X_test.copy()
for col in IDS_CRUDOS:
    X_cb[col] = df_train[col].astype(str).to_numpy()
    X_test_cb[col] = df_test[col].astype(str).to_numpy()
CATEGORICAS_CB = [c for c in COLUMNAS_CB if c in X and isinstance(X[c].dtype, pd.CategoricalDtype)]
X_cb, X_test_cb = como_texto(X_cb, CATEGORICAS_CB), como_texto(X_test_cb, CATEGORICAS_CB)
tiempos["b) feature engineering"] = time.time() - inicio
print(f"Variables sin target: {X.shape[1]} | categóricas del CatBoost: {CATEGORICAS_CB} + {IDS_CRUDOS}")

Variables sin target: 156 | categóricas del CatBoost: ['listing_type_id', 'buying_mode', 'currency_id', 'status', 'seller_address_state_name', 'shipping_mode', 'unico_duenio', 'estado_inmueble', 'genero', 'temporada'] + ['seller_id', 'category_id', 'seller_address_city_name']


## c) Variables con target: submodelos y target encodings out-of-fold

Ninguna fila usa su propio target:
- **Target encoding** del vendedor y de la categoría (`te_vendedor`, `te_categoria`): % de usadas suavizado hacia el promedio, (usadas + m · prior) / (publicaciones + m) con m = 3, y la cantidad de publicaciones.
- **Submodelos de texto:** TF-IDF de palabras y caracteres + regresión logística sobre el título (C = 1) y la garantía (C = 4) normalizados; dan `proba_used_titulo` y `proba_used_warranty`.
- **Categorías vecinas** (`te_categorias_vecinas`): % de usadas de las 10 categorías más parecidas por sus títulos, ponderado por similitud y publicaciones, con m = 3. Ayuda en las categorías con pocas publicaciones.

Cada una se calcula en versión anidada (un juego por fold externo) y estándar. Los submodelos anidados son lo más lento de esta parte (~18 min).

In [4]:
inicio = time.time()
calc_vendedor = calculador_target_encoding(df_train["seller_id"], df_test["seller_id"], y, "vendedor", m=M)
calc_categoria = calculador_target_encoding(df_train["category_id"], df_test["category_id"], y, "categoria", m=M)
textos = {nombre: (df_train[col].map(normalizar_texto), df_test[col].map(normalizar_texto))
          for nombre, col in [("titulo", "title"), ("warranty", "warranty")]}
calc_titulo = calculador_submodelo(submodelo_tfidf(CONF["c_submodelo_titulo"]), *textos["titulo"], y, "proba_used_titulo")
calc_warranty = calculador_submodelo(submodelo_tfidf(CONF["c_submodelo_warranty"]), *textos["warranty"], y, "proba_used_warranty")
calc_vecinas = calculador_te_categorias_vecinas(df_train["category_id"], df_test["category_id"], CAT_VECINAS, y, m=M)

CALCULADORES = [calc_titulo, calc_warranty, calc_vendedor, calc_categoria, calc_vecinas]
anidadas = combinar_anidadas(*[variables_anidadas(c, y) for c in CALCULADORES])
estandar = [variables_estandar(c, y) for c in CALCULADORES]
ESTANDAR_TRAIN = pd.concat([e[0] for e in estandar], axis=1).set_index(X.index)
ESTANDAR_TEST = pd.concat([e[1] for e in estandar], axis=1).set_index(X_test.index)
tiempos["c) variables con target (anidadas y estándar)"] = time.time() - inicio
print(f"Variables con target: {list(ESTANDAR_TRAIN.columns)}")

Variables con target: ['proba_used_titulo', 'proba_used_warranty', 'te_vendedor', 'conteo_vendedor_train', 'te_categoria', 'conteo_categoria_train', 'te_categorias_vecinas']


## d) Predicciones de los modelos entrenados en GPU

Los tres fine-tunings de BETO (título; título + garantía; BETO cased) y TabICLv2 se entrenaron en Kaggle con GPU (notebooks en `kaggle_gpu/`). **No se reentrenan:** el entrenamiento en GPU no es determinístico bit a bit (operaciones de punto flotante en paralelo cuyo orden cambia entre corridas), así que reentrenarlos cambiaría algunas probabilidades y, con ellas, algunas filas del resultado. Se usan sus predicciones guardadas:
- **Fine-tunings:** el promedio de los tres entra como variable del LightGBM (`proba_used_finetune_promedio`). En train son out-of-fold con los mismos folds y en test, del modelo ajustado con todo train.
- **TabICLv2:** out-of-fold (las mismas matrices de la CV anidada del LightGBM 15b) y test ajustado con todo train. Es el tercer miembro del promedio.

Se verifica que las filas y los folds coinciden con `obtener_folds(y)`.

In [5]:
def leer_gpu(nombre):
    oof = pd.read_csv(CARPETA_GPU / f"{nombre}_oof.csv").sort_values("fila")
    test = pd.read_csv(CARPETA_GPU / f"{nombre}_test.csv").sort_values("fila")
    assert (oof["fila"].to_numpy() == np.arange(len(Y))).all() and (test["fila"].to_numpy() == np.arange(len(X_test))).all()
    assert (oof["fold"].to_numpy() == FOLD).all(), f"{nombre}: los folds no coinciden con obtener_folds(y)"
    assert oof["proba_used"].notna().all() and test["proba_used"].notna().all()
    return oof["proba_used"].to_numpy(), test["proba_used"].to_numpy()


for variable, archivo in CONF["finetunes"].items():
    X[variable], X_test[variable] = leer_gpu(f"finetune_{archivo}")
X["proba_used_finetune_promedio"] = X[list(CONF["finetunes"])].mean(axis=1)
X_test["proba_used_finetune_promedio"] = X_test[list(CONF["finetunes"])].mean(axis=1)
TABICL_OOF, TABICL_TEST = leer_gpu("tabicl_top100")
print("Predicciones de GPU verificadas: filas y folds iguales a obtener_folds(y)")

Predicciones de GPU verificadas: filas y folds iguales a obtener_folds(y)


## e) LightGBM y CatBoost

- **LightGBM** (164 variables): hiperparámetros elegidos con Optuna (modelo 06), una semilla (42), hasta 10.000 árboles con early stopping de 200 rondas sobre el fold de validación.
- **CatBoost con ids crudos** (123 variables): profundidad 6, learning rate 0,05, semilla 42, early stopping de 200.

Para cada modelo: CV anidada (predicciones out-of-fold para los pesos) y CV estándar, cuyos 5 modelos promediados dan la predicción de test.

In [6]:
def crear_lgbm():
    return lgb.LGBMClassifier(**CONF["lightgbm"]["parametros"], random_state=CONF["semilla"], bagging_seed=CONF["semilla"])


def fit_params_lgbm(X_val, y_val):
    return {"eval_X": X_val, "eval_y": y_val, "callbacks": [lgb.early_stopping(CONF["lightgbm"]["early_stopping"], verbose=False)]}


inicio = time.time()
print("=== LightGBM, CV anidada")
lgbm_anidada = evaluar_modelo_anidado(crear_lgbm, X, anidadas, y, COLUMNAS_LGBM, fit_params=fit_params_lgbm)
tiempos["e) LightGBM, CV anidada"] = time.time() - inicio
inicio = time.time()
print("=== LightGBM, CV estándar (test)")
X_lgbm, X_test_lgbm = X.join(ESTANDAR_TRAIN)[COLUMNAS_LGBM], X_test.join(ESTANDAR_TEST)[COLUMNAS_LGBM]
lgbm_estandar = evaluar_modelo(crear_lgbm, X_lgbm, y, X_test_lgbm, fit_params=fit_params_lgbm)
tiempos["e) LightGBM, CV estándar"] = time.time() - inicio

=== LightGBM, CV anidada


Fold 1: accuracy = 0.9468 | AUC = 0.9875 | iteraciones = 3212


Fold 2: accuracy = 0.9439 | AUC = 0.9878 | iteraciones = 3858


Fold 3: accuracy = 0.9479 | AUC = 0.9883 | iteraciones = 4149


Fold 4: accuracy = 0.9476 | AUC = 0.9881 | iteraciones = 3019


Fold 5: accuracy = 0.9470 | AUC = 0.9881 | iteraciones = 2914
Accuracy CV anidada: 0.9467 ± 0.0016 | AUC: 0.9880
=== LightGBM, CV estándar (test)


Fold 1: accuracy = 0.9465 | AUC = 0.9874 | iteraciones = 2632


Fold 2: accuracy = 0.9439 | AUC = 0.9876 | iteraciones = 3437


Fold 3: accuracy = 0.9458 | AUC = 0.9880 | iteraciones = 3277


Fold 4: accuracy = 0.9474 | AUC = 0.9879 | iteraciones = 3050


Fold 5: accuracy = 0.9471 | AUC = 0.9880 | iteraciones = 3196
Accuracy CV: 0.9461 ± 0.0014 | AUC CV: 0.9878


In [7]:
def crear_catboost():
    return CatBoostClassifier(**CONF["catboost"]["parametros"], cat_features=CATEGORICAS_CB + IDS_CRUDOS,
                              allow_writing_files=False)


def fit_params_catboost(X_val, y_val):
    return {"eval_set": (X_val, y_val), "early_stopping_rounds": CONF["catboost"]["early_stopping"]}


inicio = time.time()
print("=== CatBoost, CV anidada")
catboost_anidada = evaluar_modelo_anidado(crear_catboost, X_cb, anidadas, y, COLUMNAS_CB, fit_params=fit_params_catboost)
tiempos["e) CatBoost, CV anidada"] = time.time() - inicio
inicio = time.time()
print("=== CatBoost, CV estándar (test)")
X_catboost, X_test_catboost = X_cb.join(ESTANDAR_TRAIN)[COLUMNAS_CB], X_test_cb.join(ESTANDAR_TEST)[COLUMNAS_CB]
catboost_estandar = evaluar_modelo(crear_catboost, X_catboost, y, X_test_catboost, fit_params=fit_params_catboost)
tiempos["e) CatBoost, CV estándar"] = time.time() - inicio

=== CatBoost, CV anidada


Fold 1: accuracy = 0.9448 | AUC = 0.9867 | iteraciones = 2207


Fold 2: accuracy = 0.9413 | AUC = 0.9863 | iteraciones = 2010


Fold 3: accuracy = 0.9439 | AUC = 0.9868 | iteraciones = 1657


Fold 4: accuracy = 0.9434 | AUC = 0.9867 | iteraciones = 1502


Fold 5: accuracy = 0.9437 | AUC = 0.9872 | iteraciones = 1668
Accuracy CV anidada: 0.9434 ± 0.0013 | AUC: 0.9867
=== CatBoost, CV estándar (test)


Fold 1: accuracy = 0.9449 | AUC = 0.9866 | iteraciones = 2179


Fold 2: accuracy = 0.9417 | AUC = 0.9863 | iteraciones = 2008


Fold 3: accuracy = 0.9444 | AUC = 0.9868 | iteraciones = 1749


Fold 4: accuracy = 0.9436 | AUC = 0.9866 | iteraciones = 1789


Fold 5: accuracy = 0.9426 | AUC = 0.9871 | iteraciones = 1263
Accuracy CV: 0.9435 ± 0.0013 | AUC CV: 0.9867


## f) Pesos del promedio

Promedio ponderado w₁ · LightGBM + w₂ · CatBoost + w₃ · TabICL, con pesos no negativos que suman 1. Se eligen minimizando el log loss de las predicciones out-of-fold de la CV anidada (SLSQP con el gradiente analítico).
- **Accuracy esperada:** pesos ajustados con 4 folds y evaluados en el quinto, rotando.
- **Pesos finales:** ajustados con los 5 folds; se aplican a las predicciones de test. Umbral 0,5.

In [8]:
NOMBRES = ["LightGBM", "CatBoost ids crudos", "TabICL top100"]
P = np.column_stack([np.asarray(lgbm_anidada["oof"]), np.asarray(catboost_anidada["oof"]), TABICL_OOF])
P_test = np.column_stack([lgbm_estandar["proba_test"], catboost_estandar["proba_test"], TABICL_TEST])

oof_combinado, pesos = np.zeros(len(Y)), {}
for k, (_, idx_val) in enumerate(folds, start=1):
    w = ajustar_pesos(np.setdiff1d(np.arange(len(Y)), idx_val), P, Y)
    pesos[f"evaluado en fold {k}"] = w
    oof_combinado[idx_val] = P[idx_val] @ w
PESOS = ajustar_pesos(np.arange(len(Y)), P, Y)
pesos["5 folds (para test)"] = PESOS
display(pd.DataFrame(pesos, index=NOMBRES).T.round(4))

combinado = resultado_desde_oof(oof_combinado, P_test @ PESOS, y)
filas = {n: resultado_desde_oof(P[:, i], P_test[:, i], y) for i, n in enumerate(NOMBRES)} | {"Promedio ponderado": combinado}
display(pd.DataFrame({n: {"accuracy anidada": r["accuracy_media"], "desvío": r["accuracy_desvio"], "AUC": r["auc"]}
                      for n, r in filas.items()}).T.round(4))
PROBA_TEST = P_test @ PESOS
print(f"% used predicho en test: {np.mean(PROBA_TEST >= 0.5):.4f}")

,LightGBM,CatBoost ids crudos,TabICL top100
evaluado en fold 1,0.6464,0.1495,0.2041
evaluado en fold 2,0.5975,0.1801,0.2224
evaluado en fold 3,0.6247,0.1821,0.1933
evaluado en fold 4,0.6153,0.1960,0.1887
evaluado en fold 5,0.6377,0.1678,0.1946
5 folds (para test),0.6243,0.1752,0.2005


,accuracy anidada,desvío,AUC
LightGBM,0.9467,0.0016,0.9880
CatBoost ids crudos,0.9434,0.0013,0.9867
TabICL top100,0.9435,0.0017,0.9870
Promedio ponderado,0.9473,0.0013,0.9882


% used predicho en test: 0.4731


## g) Guardado de los modelos

En `modelos/` queda lo que usa `2_prediccion.ipynb`:
- los 5 LightGBM (`lightgbm_fold_k.txt`) y los 5 CatBoost (`catboost_fold_k.cbm`) de la CV estándar, en el formato nativo de cada librería (la predicción de test es el promedio de cada grupo);
- los pesos del promedio (`pesos.json`);
- las matrices de test de cada modelo, solo para que el notebook 2 verifique que las que rearma son iguales.

In [9]:
import json

CARPETA_MODELOS.mkdir(exist_ok=True)
for k, (modelo_lgbm, modelo_cb) in enumerate(zip(lgbm_estandar["modelos"], catboost_estandar["modelos"]), start=1):
    modelo_lgbm.booster_.save_model(CARPETA_MODELOS / f"lightgbm_fold_{k}.txt")  # guarda los árboles hasta la mejor iteración
    modelo_cb.save_model(str(CARPETA_MODELOS / f"catboost_fold_{k}.cbm"))
X_test_lgbm.to_pickle(CARPETA_MODELOS / "X_test_lightgbm.pkl.gz")
X_test_catboost.to_pickle(CARPETA_MODELOS / "X_test_catboost.pkl.gz")
with open(CARPETA_MODELOS / "pesos.json", "w", encoding="utf-8") as f:
    json.dump({"modelos": NOMBRES, "pesos": PESOS.tolist(), "umbral": 0.5}, f, indent=2)
display(pd.Series({p.name: p.stat().st_size / 1e6 for p in sorted(CARPETA_MODELOS.iterdir()) if p.is_file() and p.name != ".gitkeep"}).round(1).to_frame("MB"))

,MB
catboost_fold_1.cbm,41.1
catboost_fold_2.cbm,38.7
catboost_fold_3.cbm,30.5
catboost_fold_4.cbm,40.9
catboost_fold_5.cbm,26.2
lightgbm_fold_1.txt,7.3
lightgbm_fold_2.txt,9.5
lightgbm_fold_3.txt,9.0
lightgbm_fold_4.txt,8.4
lightgbm_fold_5.txt,8.8


In [10]:
tiempos["total"] = time.time() - inicio_total
display(pd.Series(tiempos).div(60).round(1).to_frame("minutos"))

,minutos
a) carga y procesamiento,0.9
b) feature engineering,1.3
c) variables con target (anidadas y estándar),23.3
"e) LightGBM, CV anidada",7.4
"e) LightGBM, CV estándar",4.3
"e) CatBoost, CV anidada",26.2
"e) CatBoost, CV estándar",24.9
total,88.6
